# Best Model - Machine Learning Regression Comparison

## Objective
This notebook implements and compares five different machine learning regression models to predict real estate property prices. Each model is trained on preprocessed data and evaluated using multiple metrics (MAE, MSE, RMSE, R²) to determine the best performer.

## Dataset
- **Source**: `dataset/cleaned_data.csv`
- **Target Variable**: `Price` (property price)
- **Features**: Year, Month, Town/City, District, County, and engineered features

## Models to Compare
1. **Linear Regression** - Baseline linear model
2. **Decision Tree Regressor** - Tree-based model
3. **Random Forest Regressor** - Ensemble of trees (200 estimators)
4. **Support Vector Regression (SVR)** - RBF kernel
5. **XGBoost Regressor** - Gradient boosting with 300 estimators

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

df = pd.read_csv('/home/sukesh/Desktop/RealtyAI/B-13-RealtyAI-Smart-Real-Estate-Insight-Platform/dataset/cleaned_data.csv')

## Import Required Libraries and Load Data

Import all necessary libraries for machine learning models, data manipulation, and evaluation metrics. Then load the cleaned dataset.

In [4]:
df.head()


,Price,Old/New,Town/City,District,County,PPDCategory Type,Property Type_D,Property Type_F,Property Type_O,Property Type_S,Property Type_T,Duration_F,Duration_L,Duration_U,Year,Month,Price_scaled,Price_minmax
0,10.126671,1,OLDHAM,OLDHAM,GREATER MANCHESTER,0,0,0,0,0,1,1,0,0,1995,8,-1.401577,0.250575
1,10.657283,1,GRAYS,THURROCK,THURROCK,0,0,0,0,1,0,1,0,0,1995,8,-0.493593,0.335777
2,10.714440,1,HIGHBRIDGE,SEDGEMOOR,SOMERSET,0,0,0,0,0,1,1,0,0,1995,6,-0.395786,0.344955
3,10.672461,1,BEDFORD,NORTH BEDFORDSHIRE,BEDFORDSHIRE,0,0,0,0,0,1,1,0,0,1995,11,-0.467621,0.338215
4,9.846917,1,WAKEFIELD,LEEDS,WEST YORKSHIRE,0,0,0,0,1,0,1,0,0,1995,6,-1.880292,0.205654


In [5]:
df["Sale_time"] = df["Year"] * 12 + df["Month"]
df.head()

,Price,Old/New,Town/City,District,County,PPDCategory Type,Property Type_D,Property Type_F,Property Type_O,Property Type_S,Property Type_T,Duration_F,Duration_L,Duration_U,Year,Month,Price_scaled,Price_minmax,Sale_time
0,10.126671,1,OLDHAM,OLDHAM,GREATER MANCHESTER,0,0,0,0,0,1,1,0,0,1995,8,-1.401577,0.250575,23948
1,10.657283,1,GRAYS,THURROCK,THURROCK,0,0,0,0,1,0,1,0,0,1995,8,-0.493593,0.335777,23948
2,10.714440,1,HIGHBRIDGE,SEDGEMOOR,SOMERSET,0,0,0,0,0,1,1,0,0,1995,6,-0.395786,0.344955,23946
3,10.672461,1,BEDFORD,NORTH BEDFORDSHIRE,BEDFORDSHIRE,0,0,0,0,0,1,1,0,0,1995,11,-0.467621,0.338215,23951
4,9.846917,1,WAKEFIELD,LEEDS,WEST YORKSHIRE,0,0,0,0,1,0,1,0,0,1995,6,-1.880292,0.205654,23946


## Feature Engineering - Temporal Features

Create a temporal feature by combining Year and Month into a single `Sale_time` variable. This captures the sequential time period (in months) when each property was sold.

In [6]:
X = df.drop("Price", axis=1)
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Train-Test Split

Split the data into training (80%) and testing (20%) sets to evaluate model performance on unseen data. Use `random_state=42` for reproducibility.

In [7]:
train_temp = X_train.copy()
train_temp["Price"] = y_train

city_mean = train_temp.groupby("Town/City")["Price"].mean()

X_train["Town_encoded"] = X_train["Town/City"].map(city_mean)
X_test["Town_encoded"] = X_test["Town/City"].map(city_mean)

global_mean = y_train.mean()
X_test["Town_encoded"] = X_test["Town_encoded"].fillna(global_mean)

## Target Encoding for Categorical Features

Encode the categorical `Town/City` feature using mean encoding (average price per city from training data). This converts categorical data into numerical features while preserving domain knowledge about price variations by location. Unknown cities in test set are filled with the global mean price.

In [8]:
X_train = X_train.drop(columns=["Town/City", "District", "County"])
X_test = X_test.drop(columns=["Town/City", "District", "County"])

X_train = X_train.drop(columns=["Price_scaled", "Price_minmax"], errors="ignore")
X_test = X_test.drop(columns=["Price_scaled", "Price_minmax"], errors="ignore")

## Feature Cleanup

Drop categorical and pre-scaled columns to prepare data for model training:
- Drop original categorical columns: `Town/City`, `District`, `County` (replaced by encoded feature)
- Drop pre-scaled price features: `Price_scaled`, `Price_minmax` (would cause data leakage)

In [9]:
lr = LinearRegression()
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_pred)
lr_mse = mean_squared_error(y_test, lr_pred)
lr_rmse = np.sqrt(lr_mse)
lr_r2 = r2_score(y_test, lr_pred)

print("Linear Regression")
print("MAE:", lr_mae)
print("MSE:", lr_mse)
print("RMSE:", lr_rmse)
print("R2:", lr_r2)

Linear Regression
MAE: 0.2904814762658569
MSE: 0.16141298038813265
RMSE: 0.40176234316836196
R2: 0.5259873728956943


## Model Training and Evaluation

Train and evaluate five different regression models using the same evaluation metrics to compare performance fairly.

### Evaluation Metrics
- **MAE (Mean Absolute Error)**: Average absolute prediction error - lower is better
- **MSE (Mean Squared Error)**: Average squared error - penalizes larger errors - lower is better
- **RMSE (Root Mean Squared Error)**: Square root of MSE - same scale as target variable - lower is better
- **R² Score**: Proportion of variance explained by model (0-1 scale) - higher is better

---

### Model 1: Linear Regression
Baseline model assuming linear relationship between features and price.

In [10]:
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

dt_mae = mean_absolute_error(y_test, dt_pred)
dt_mse = mean_squared_error(y_test, dt_pred)
dt_rmse = np.sqrt(dt_mse)
dt_r2 = r2_score(y_test, dt_pred)

print("Decision Tree")
print("MAE:", dt_mae)
print("MSE:", dt_mse)
print("RMSE:", dt_rmse)
print("R2:", dt_r2)

Decision Tree
MAE: 0.324807855521491
MSE: 0.19812528777108993
RMSE: 0.4451126686256974
R2: 0.4181763577728004


### Model 2: Decision Tree Regressor
Tree-based model that recursively splits features to minimize prediction error. Can capture non-linear relationships but may overfit.

In [11]:
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_mse = mean_squared_error(y_test, rf_pred)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest")
print("MAE:", rf_mae)
print("MSE:", rf_mse)
print("RMSE:", rf_rmse)
print("R2:", rf_r2)

Random Forest
MAE: 0.300350260538588
MSE: 0.16894655544404183
RMSE: 0.4110310881722231
R2: 0.5038639371276902


### Model 3: Random Forest Regressor
Ensemble of 200 decision trees that reduces overfitting through averaging. Better generalization than single tree through democratic voting.

In [12]:
svr = SVR(kernel="rbf")
svr.fit(X_train, y_train)

svr_pred = svr.predict(X_test)

svr_mae = mean_absolute_error(y_test, svr_pred)
svr_mse = mean_squared_error(y_test, svr_pred)
svr_rmse = np.sqrt(svr_mse)
svr_r2 = r2_score(y_test, svr_pred)

print("SVR")
print("MAE:", svr_mae)
print("MSE:", svr_mse)
print("RMSE:", svr_rmse)
print("R2:", svr_r2)

SVR
MAE: 0.4391659274706888
MSE: 0.341695092010533
RMSE: 0.584546911727821
R2: -0.003437194723084902


### Model 4: Support Vector Regression (SVR)
Uses RBF (Radial Basis Function) kernel to map features into higher-dimensional space. Excellent for capturing complex non-linear patterns.

In [15]:
model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

model.fit(X_train, y_train)

xgb_pred = model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_mse = mean_squared_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(xgb_mse)
xgb_r2 = r2_score(y_test, xgb_pred)

print("XGBoost")
print("MAE:", xgb_mae)
print("MSE:", xgb_mse)
print("RMSE:", xgb_rmse)
print("R2:", xgb_r2)

XGBoost
MAE: 0.27862612023422056
MSE: 0.14912965830869607
RMSE: 0.3861730937140702
R2: 0.5620591296679281


### Model 5: XGBoost Regressor
Gradient boosting algorithm with sequential tree learning. Each tree corrects errors from previous trees.
- **n_estimators**: 300 boosting rounds
- **learning_rate**: 0.05 (controls feature contribution)
- **max_depth**: 6 (controls tree complexity)

Often provides state-of-the-art performance through iterative improvement.

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression","Decision Tree","Random Forest","SVR","XGBoost"],
    "MAE": [lr_mae, dt_mae, rf_mae, svr_mae, xgb_mae],
    "MSE": [lr_mse, dt_mse, rf_mse, svr_mse, xgb_mse],
    "RMSE": [lr_rmse, dt_rmse, rf_rmse, svr_rmse, xgb_rmse],
    "R2 Score": [lr_r2, dt_r2, rf_r2, svr_r2, xgb_r2]
})

results

,Model,MAE,MSE,RMSE,R2 Score
0,Linear Regression,0.290481,0.161413,0.401762,0.525987
1,Decision Tree,0.324808,0.198125,0.445113,0.418176
2,Random Forest,0.300350,0.168947,0.411031,0.503864
3,SVR,0.439166,0.341695,0.584547,-0.003437
4,XGBoost,0.278626,0.149130,0.386173,0.562059


## Results Comparison

Create a comprehensive DataFrame comparing all five models across all evaluation metrics. This summary table makes it easy to identify which model performs best and by how much.

## Conclusions and Next Steps

### Key Findings from Model Comparison
- Compare MAE, MSE, RMSE, and R² scores from the results table above
- Identify which model has the lowest error metrics and highest R² score
- Consider trade-offs between model complexity and performance

### Typical Expected Results
- **XGBoost** typically achieves best performance with carefully tuned hyperparameters
- **Random Forest** provides strong generalization with solid ensemble benefits
- **SVR** captures complex non-linear patterns effectively
- **Decision Tree** is prone to overfitting but highly interpretable
- **Linear Regression** serves as baseline for understanding linear component

### Potential Next Steps
1. **Hyperparameter Tuning**: Use GridSearchCV or RandomizedSearchCV for optimization
2. **Feature Importance**: Analyze feature importance from tree-based and XGBoost models
3. **Cross-Validation**: Implement k-fold cross-validation for more robust evaluation
4. **Model Ensemble**: Combine predictions from multiple models for improved performance
5. **Production Deployment**: Save the best model and create prediction pipeline
6. **Additional Preprocessing**: Explore more feature engineering techniques
7. **Outlier Analysis**: Investigate predictions with highest errors for data quality insights